# Full-population provenance trace (no active-provider pre-filter)

**Author**: Sian Teesdale  
**Date created**: 3rd July 2026  
**Dataset Scope**: all datasets with any `quality=some` entity owned by a non-government organisation  
**Purpose**: `1_find_flagged_entities.ipynb` only looks at entities whose owning organisation currently has a **registered, live endpoint** in `source_pipeline` for that specific dataset (an "active provider"). That filter matched the original ticket's literal ask, but it also means an entity is invisible to notebooks 1-3 if its owning organisation has *never* registered as a provider for that dataset at all — even if the entity's real data can still be traced to someone else's active submission.

The clearest example found while investigating: **Northumberland National Park Authority** owns 231 `listed-building-outline` entities, but has **no `source_pipeline` entry at all** for that dataset (not inactive/end-dated — simply never registered). Notebook 1 would drop these before ever checking their quality or provenance. Yet the entities' real data traces cleanly to Northumberland County Council's own active submission.

This notebook removes that pre-filter: it starts from *every* `quality=some` entity owned by a `local-authority:`/`national-park-authority:`/`development-corporation:` organisation, active provider or not, and runs the same provenance trace and classification as `3_trace_true_provenance.ipynb`. It keeps an `is_active_provider` column throughout so the two populations (already covered by notebooks 1-3, vs newly reachable only here) stay distinguishable in the output.

**Trace chain and classification are identical to notebook 3** — see that notebook for the full writeup of the three legitimate look-alike patterns (government seeding, regional aggregation, Local Government Reorganisation succession) that get excluded from `confirmed_misattribution`.

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

from helpers import chunk, datasette_sql, fetch_filtered_table_csv, fetch_table_csv, parallel_fetch

NON_GOV_PREFIXES = ("local-authority:", "national-park-authority:", "development-corporation:")
GOV_PREFIX = "government-organisation:"
FANOUT_THRESHOLD = 10  # true-orgs serving more than this many distinct assigned orgs are treated as legitimate aggregators (e.g. GLA), not bugs

DATA_DIR = os.path.join("..", "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Build the full non-government `quality=some` population

Every dataset, every `local-authority:`/`national-park-authority:`/`development-corporation:` owner — no active-provider filter. `is_active_provider` is still computed and kept as a column so the two populations stay distinguishable downstream.

In [2]:
print("Fetching organisation, source and source_pipeline tables...")
org_df = fetch_table_csv("organisation")
entity_to_org = org_df.set_index("entity")["organisation"].to_dict()
org_to_name = org_df.set_index("organisation")["name"].to_dict()
org_active = (org_df["end_date"].isna()).set_axis(org_df["organisation"]).to_dict()

source = fetch_table_csv("source")
source_pipeline = fetch_table_csv("source_pipeline")
merged_source = source.merge(source_pipeline, on="source")
active_source = merged_source[merged_source["endpoint"] != ""]
active_pairs = set(zip(active_source["pipeline"], active_source["organisation"]))
print(f"  {len(active_pairs)} active (dataset, organisation) pairs, all org types")

all_datasets = datasette_sql("digital-land", "SELECT dataset FROM dataset ORDER BY dataset")["dataset"].tolist()
print(f"  {len(all_datasets)} total datasets on platform")

Fetching organisation, source and source_pipeline tables...


  3718 active (dataset, organisation) pairs, all org types
  279 total datasets on platform


In [3]:
def _some_quality_for_dataset(dataset):
    try:
        df = fetch_filtered_table_csv(
            dataset, "entity",
            columns=["name", "reference", "organisation_entity", "quality"],
            quality="some",
        )
    except Exception:
        return None
    if df.empty:
        return None
    df["dataset"] = dataset
    return df


print(f"Querying quality='some' entities across all {len(all_datasets)} datasets (parallel)...")
some_quality_parts = []
with ThreadPoolExecutor(max_workers=25) as pool:
    futures = {pool.submit(_some_quality_for_dataset, ds): ds for ds in all_datasets}
    for f in as_completed(futures):
        r = f.result()
        if r is not None:
            some_quality_parts.append(r)

all_some = pd.concat(some_quality_parts, ignore_index=True)
all_some["organisation_entity"] = pd.to_numeric(all_some["organisation_entity"], errors="coerce")
all_some = all_some.dropna(subset=["organisation_entity"]).copy()
all_some["organisation_entity"] = all_some["organisation_entity"].astype(int)
all_some["organisation"] = all_some["organisation_entity"].map(entity_to_org)
all_some = all_some.dropna(subset=["organisation"])
all_some["org_prefix"] = all_some["organisation"].str.split(":").str[0] + ":"

flagged_df = all_some[all_some["org_prefix"].isin(NON_GOV_PREFIXES)].copy()
flagged_df["organisation_name"] = flagged_df["organisation"].map(org_to_name)
flagged_df["entity_url"] = "https://www.planning.data.gov.uk/entity/" + flagged_df["entity"].astype(str)
flagged_df["is_active_provider"] = flagged_df.apply(
    lambda r: (r["dataset"], r["organisation"]) in active_pairs, axis=1
)

print(f"\n{len(flagged_df)} total flagged entities across {flagged_df['dataset'].nunique()} datasets")
print(flagged_df["is_active_provider"].value_counts())

Querying quality='some' entities across all 279 datasets (parallel)...



45639 total flagged entities across 27 datasets
is_active_provider
False    37885
True      7754
Name: count, dtype: int64


## 2. Fetch facts for every flagged entity, per dataset

Batched via `entity__in=(...)` (chunks of 300 ids), same pattern as notebook 3.

In [4]:
fact_jobs = []
for dataset, group in flagged_df.groupby("dataset"):
    entity_ids = group["entity"].astype(int).tolist()
    for batch in chunk(entity_ids, 300):
        fact_jobs.append((
            fetch_filtered_table_csv,
            (dataset, "fact"),
            {"columns": ["entity"], "entity__in": ",".join(map(str, batch))},
        ))

print(f"Fetching facts for {len(flagged_df)} entities across {len(fact_jobs)} batched requests...")
fact_parts = parallel_fetch(fact_jobs)
facts_df = pd.concat(fact_parts, ignore_index=True) if fact_parts else pd.DataFrame(columns=["fact", "entity"])
facts_df = facts_df.merge(flagged_df[["dataset", "entity"]].drop_duplicates(), on="entity", how="left")
print(f"{len(facts_df)} fact rows, {facts_df['fact'].nunique()} distinct facts")

Fetching facts for 45639 entities across 169 batched requests...


  (10 batch requests failed -- retrying once at lower concurrency)


633449 fact rows, 439412 distinct facts


## 3. Resolve facts to resources, then resources to endpoints

Same two-step lineage as notebook 3 (`fact_resource`'s PK is `rowid`, not `fact`, so `fact` must be requested explicitly; `log` resolves resource -> endpoint, deduped since one resource shows up once per fetch attempt).

In [5]:
fr_jobs = []
for dataset, group in facts_df.groupby("dataset"):
    fact_hashes = group["fact"].unique().tolist()
    for batch in chunk(fact_hashes, 80):  # fact hashes are 64 chars -- larger batches hit HTTP 414
        fr_jobs.append((
            fetch_filtered_table_csv,
            (dataset, "fact_resource"),
            {"columns": ["fact", "resource"], "fact__in": ",".join(batch)},
        ))

print(f"Resolving facts to resources across {len(fr_jobs)} batched requests...")
fr_parts = parallel_fetch(fr_jobs)
fr_df = pd.concat(fr_parts, ignore_index=True) if fr_parts else pd.DataFrame(columns=["fact", "resource"])
print(f"{len(fr_df)} fact_resource rows, {fr_df['resource'].nunique()} distinct resources")

Resolving facts to resources across 6751 batched requests...


1579115 fact_resource rows, 831 distinct resources


In [6]:
resource_hashes = fr_df["resource"].unique().tolist()
log_jobs = [
    (
        fetch_filtered_table_csv,
        ("digital-land", "log"),
        {"columns": ["resource", "endpoint"], "resource__in": ",".join(batch)},
    )
    for batch in chunk(resource_hashes, 80)
]

print(f"Resolving {len(resource_hashes)} resources to endpoints across {len(log_jobs)} batched requests...")
log_parts = parallel_fetch(log_jobs)
log_df = (
    pd.concat(log_parts, ignore_index=True).drop_duplicates(subset=["resource", "endpoint"])
    if log_parts else pd.DataFrame(columns=["resource", "endpoint"])
)
print(f"{len(log_df)} distinct (resource, endpoint) pairs")

Resolving 831 resources to endpoints across 11 batched requests...


841 distinct (resource, endpoint) pairs


## 4. Join the full chain and derive the true organisation per entity

In [7]:
endpoint_df = fetch_table_csv("endpoint")
source_df = fetch_table_csv("source")[["endpoint", "organisation"]].drop_duplicates()

chain = (
    facts_df.merge(fr_df, on="fact")
    .merge(log_df, on="resource")
    .merge(source_df, on="endpoint")
    .merge(endpoint_df[["endpoint", "endpoint_url"]], on="endpoint")
)
print(f"{len(chain)} chain rows linking entity -> true organisation")

true_org_per_entity = (
    chain.groupby("entity")["organisation"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="true_orgs")
)
true_endpoint_per_entity = (
    chain.groupby("entity")["endpoint_url"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="true_endpoint_urls")
)

result = flagged_df.merge(true_org_per_entity, on="entity", how="left")
result = result.merge(true_endpoint_per_entity, on="entity", how="left")

2399881 chain rows linking entity -> true organisation


## 5. Classify

Identical three-stage logic to notebook 3:
1. Government-organisation contributors -> `seeded_by_government` (Historic England, MHCLG).
2. A true-org serving more than `FANOUT_THRESHOLD` distinct assigned orgs in the dataset -> `seeded_by_regional_body` (e.g. GLA).
3. A true-org with an `end_date` -> `succeeded_inactive_org` (Local Government Reorganisation, e.g. SNR -> WNUA).

What's left after all three exclusions is `confirmed_misattribution`.

In [8]:
def classify_stage1(row):
    true_orgs = row["true_orgs"]
    assigned = row["organisation"]
    if not isinstance(true_orgs, list) or len(true_orgs) == 0:
        return "no_resource_found"
    if assigned in true_orgs:
        return "consistent"
    non_gov = [o for o in true_orgs if not o.startswith(GOV_PREFIX)]
    if len(non_gov) == 0:
        return "seeded_by_government"
    if len(non_gov) == 1:
        return "confirmed_misattribution"
    return "ambiguous"


result["classification"] = result.apply(classify_stage1, axis=1)
result["true_org"] = result["true_orgs"].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else None)

# Stage 2: reclassify high-fan-out true-orgs (regional aggregators like GLA) out of confirmed_misattribution.
fanout = (
    result[result["classification"] == "confirmed_misattribution"]
    .groupby(["dataset", "true_org"])["organisation"]
    .nunique()
)
high_fanout_pairs = set(fanout[fanout > FANOUT_THRESHOLD].index)

result.loc[
    result.apply(lambda r: (r["dataset"], r["true_org"]) in high_fanout_pairs, axis=1)
    & (result["classification"] == "confirmed_misattribution"),
    "classification",
] = "seeded_by_regional_body"

# Stage 3: reclassify cases where the true_org is itself inactive (has an end_date) --
# a Local Government Reorganisation succession, not a genuine mix-up between two active peers.
result["true_org_active"] = result["true_org"].map(org_active)
result.loc[
    (result["classification"] == "confirmed_misattribution") & (result["true_org_active"] == False),
    "classification",
] = "succeeded_inactive_org"

result["classification"].value_counts()

classification
seeded_by_government        41234
seeded_by_regional_body      2124
succeeded_inactive_org       1313
ambiguous                     499
confirmed_misattribution      446
no_resource_found              23
Name: count, dtype: int64

## 6. Breakdown tables

In [9]:
# Split by whether the assigned org was already an active provider -- i.e. whether
# notebooks 1-3 would have caught this entity at all, or whether it's only reachable here.
result.groupby(["is_active_provider", "classification"]).size().unstack(fill_value=0)

classification,ambiguous,confirmed_misattribution,no_resource_found,seeded_by_government,seeded_by_regional_body,succeeded_inactive_org
is_active_provider,,,,,,
False,0,273,0,36399,0,1213
True,499,173,23,4835,2124,100


In [10]:
confirmed = result[result["classification"] == "confirmed_misattribution"]
by_org_pair = (
    confirmed.groupby(["dataset", "organisation", "true_org", "is_active_provider"])
    .agg(entities=("entity", "nunique"))
    .sort_values("entities", ascending=False)
)
by_org_pair

entities
dataset                 organisation                      true_org                          is_active_provider          
listed-building-outline national-park-authority:Q72617890 local-authority:NBL               False                    231
brownfield-land         local-authority:WOK               local-authority:WOI               True                      77
developer-agreement     national-park-authority:Q72617158 local-authority:NEW               True                      44
plan-timetable          local-authority:TOR               national-park-authority:Q72617669 True                      12
brownfield-land         local-authority:CHW               local-authority:CHE               True                       7
developer-agreement     local-authority:WYC               local-authority:WOR               True                       5
brownfield-land         local-authority:SHR               local-authority:CHE               True                       4
developer-agreement     local-authority:BRM               local-authority:WOR               True                       3
                        local-authority:MAV               local-authority:WOR               True                       3
article-4-direction     local-authority:FEN               local-authority:FOE               True                       2
conservation-area       national-park-authority:Q20198711 local-authority:EHA               False                      2
developer-agreement     local-authority:WYE               local-authority:WOR               True                       2
brownfield-land         local-authority:CON               local-authority:CHE               True                       1
conservation-area       national-park-authority:Q4972284  local-authority:BRO               False                      1

In [11]:
confirmed[["dataset", "entity", "organisation", "true_org", "is_active_provider", "entity_url", "true_endpoint_urls"]].head(30)

,dataset,entity,organisation,true_org,is_active_provider,entity_url,true_endpoint_urls
0,article-4-direction,6102345,local-authority:FEN,local-authority:FOE,True,https://www.planning.data.gov.uk/entity/6102345,[https://www.fdean.gov.uk/media/smod3gr1/artic...
1,article-4-direction,6102346,local-authority:FEN,local-authority:FOE,True,https://www.planning.data.gov.uk/entity/6102346,[https://www.fdean.gov.uk/media/smod3gr1/artic...
659,brownfield-land,1736757,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736757,[https://www.woking2027.info/ldfregisters/brow...
660,brownfield-land,1736758,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736758,[https://www.woking2027.info/ldfregisters/brow...
661,brownfield-land,1736759,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736759,[https://www.woking2027.info/ldfregisters/brow...
662,brownfield-land,1736760,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736760,[https://www.woking2027.info/ldfregisters/brow...
663,brownfield-land,1736761,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736761,[https://www.woking2027.info/ldfregisters/brow...
664,brownfield-land,1736762,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736762,[https://www.woking2027.info/ldfregisters/brow...
665,brownfield-land,1736763,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736763,[https://www.woking2027.info/ldfregisters/brow...
666,brownfield-land,1736764,local-authority:WOK,local-authority:WOI,True,https://www.planning.data.gov.uk/entity/1736764,[https://www.woking2027.info/ldfregisters/brow...


## 7. Export prioritized list

In [12]:
priority_order = {
    "confirmed_misattribution": 0,
    "ambiguous": 1,
    "no_resource_found": 2,
    "succeeded_inactive_org": 3,
    "seeded_by_regional_body": 4,
    "seeded_by_government": 5,
    "consistent": 6,
}
result["_priority"] = result["classification"].map(priority_order)
result = result.sort_values(["_priority", "dataset", "organisation", "entity"]).drop(columns="_priority")

export_cols = [
    "dataset", "entity", "name", "reference", "organisation", "organisation_name",
    "quality", "is_active_provider", "entity_url", "classification", "true_org", "true_orgs", "true_endpoint_urls",
]
out_path = os.path.join(DATA_DIR, "flagged_entities_full_population_with_provenance.csv")
result[export_cols].to_csv(out_path, index=False)
print(f"Saved {len(result)} rows to {out_path}")

Saved 45639 rows to ../../data/flagged_entities_full_population_with_provenance.csv
